# Semantic Extractor Training

Enable GPU, attach the private semantic training dataset, and run all cells.
The notebook writes only safe artifacts to `/kaggle/working/`.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys
import tempfile
import urllib.request
import zipfile

repo_zip = Path('/kaggle/working/data_analysis_LLM.zip')
repo_root = Path('/kaggle/working/data_analysis_LLM')
urllib.request.urlretrieve('https://github.com/PritishMete/data_analysis_LLM/archive/refs/heads/main.zip', repo_zip)
with tempfile.TemporaryDirectory(dir='/kaggle/working') as temp_dir:
    with zipfile.ZipFile(repo_zip, 'r') as archive:
        archive.extractall(temp_dir)
    extracted = next(Path(temp_dir).glob('data_analysis_LLM-*'))
    if repo_root.exists():
        shutil.rmtree(repo_root)
    if extracted.is_dir():
        shutil.move(str(extracted), str(repo_root))
subprocess.run([sys.executable, str(repo_root / 'kaggle' / 'bootstrap_environment.py'), '--output-root', '/kaggle/working'], check=True)
bootstrap_report = json.loads(Path('/kaggle/working/reports/dependency_install_result.json').read_text())
bootstrap_pid = bootstrap_report.get('bootstrap_pid') or 0
training = subprocess.run([sys.executable, str(repo_root / 'kaggle' / 'execute_smoke_training.py'), '--output-root', '/kaggle/working', '--bootstrap-pid', str(bootstrap_pid)], check=True)
raise SystemExit(training.returncode)